In [3]:
#Load Dataset
import pandas as pd
import numpy as np

import kagglehub
from pathlib import Path

# Massive Rotten Tomatoes Movies and Reviews, downloaded once to the kagglehub cache
RT_DIR = Path(kagglehub.dataset_download("andrezaza/clapper-massive-rotten-tomatoes-movies-and-reviews"))
movies = pd.read_csv(RT_DIR / "rotten_tomatoes_movies.csv", usecols=["id", "title", "audienceScore"]).dropna()
reviews = pd.read_csv(RT_DIR / "rotten_tomatoes_movie_reviews.csv", usecols=["id", "reviewText"]).dropna()

# Same layout as before: Title, Rating (0-10, from the audience score) and Review
dataset = (
    reviews.merge(movies, on="id")
    .assign(Rating=lambda d: d["audienceScore"] / 10)
    .rename(columns={"title": "Title", "reviewText": "Review"})[["Title", "Rating", "Review"]]
    .sample(10000, random_state=42)
    .reset_index(drop=True)
)
dataset = dataset.dropna().head(1000)

dataset.head()

,Title,Rating,Review
0,The Idea of You,6.4,"This film, as well as the reaction to it, is a..."
1,Kingdom of the Planet of the Apes,7.3,"I'm a big fan of all the planet of the apes, a..."
2,Unfrosted,5.5,Pretty much the worst criticism you can lay on...
3,The Fall Guy,7.3,Just got out of the Austin premier at SXSW and...
4,Challengers,7.7,This is a tough one. I liked the concept and t...


In [4]:
# Preprocess Data
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

eng_stopwords = stopwords.words('english')
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_sentence(sentence):
    sentence = sentence.lower()
    sentence = word_tokenize(sentence)
    sentence = [word for word in sentence if word not in eng_stopwords]
    sentence = [word for word in sentence if word.isalpha()]
    sentence = [stemmer.stem(word) for word in sentence]
    sentence = [lemmatizer.lemmatize(word) for word in sentence]
    
    return sentence

In [5]:
# Extract Features
def extract_features(review):
    review = preprocess_sentence(review)
    return {word: True for word in review if word in review} # {'kata': True}

In [6]:
# Extract Data
X = dataset['Review']
y = ['Positive' if rating > 5 else 'Negative' for rating in dataset['Rating']]

all_reviews = ''.join(X)
all_tokens = preprocess_sentence(all_reviews)

In [7]:
feature_sets = [(extract_features(review), rating) for review, rating in zip(X, y)]
feature_sets[:10]

[({'film': True,
   'well': True,
   'reaction': True,
   'wonder': True,
   'exampl': True,
   'modern': True,
   'hypocrisi': True,
   'stori': True,
   'adult': True,
   'woman': True,
   'attract': True,
   'young': True,
   'guy': True,
   'year': True,
   'younger': True,
   'enough': True,
   'son': True,
   'see': True,
   'boy': True,
   'band': True,
   'concert': True,
   'peopl': True,
   'start': True,
   'movi': True,
   'trailer': True,
   'beyond': True,
   'get': True,
   'prais': True,
   'destruct': True,
   'stigma': True,
   'happi': True,
   'remain': True,
   'age': True,
   'man': True,
   'avail': True,
   'anyon': True,
   'u': True,
   'judg': True,
   'someon': True,
   'el': True,
   'girl': True,
   'would': True,
   'disgust': True,
   'creep': True,
   'find': True,
   'way': True,
   'overcom': True,
   'midlif': True,
   'crisi': True,
   'preadatori': True,
   'pursu': True,
   'daughter': True},
  'Positive'),
 ({'big': True,
   'fan': True,
   'plan

In [8]:
def predict_review(review):
    review = extract_features(review)
    return classifier.classify(review)

In [9]:
predict_review("I hate this movie")
# Pas UAP yang penting modelnya bisa jalan, hasil bener apa engga gapapa

NameError: name 'classifier' is not defined

In [ ]:
from nltk.classify import NaiveBayesClassifier

def train_model():
    
    # Extract Data
    X = dataset['Review']
    y = ['Positive' if rating > 5 else 'Negative' for rating in dataset['Rating']]

    all_reviews = ''.join(X)
    all_tokens = preprocess_sentence(all_reviews)
    
    # Split Data
    train_count = int(len(feature_sets) * 0.8)
    train_set = feature_sets[:train_count]
    test_set = feature_sets[train_count:]

    # Classify
    global classifier
    classifier = NaiveBayesClassifier.train(train_set)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()

def get_similarity_count(query, target):
    target_v = vectorizer.fit_transform(target)
    query_v = vectorizer.transform([query])
    return cosine_similarity(query_v, target_v)

def get_top_movies(query):
    similarity = get_similarity_count(query, dataset['Review'])
    
    data = {'title': dataset['Title'], 'review': dataset['Review'], 'similarity': similarity[0]}
    
    df = pd.DataFrame(data).sort_values(by='similarity', ascending=False).head(2)
    print(df)

In [ ]:
get_top_movies('I love this movie')

                    title                                             review  \
357            Cinderella  I LOVED THIS MOVIE. There Is no reasonable exp...   
316  Call Me by Your Name  Call Me By Your Name is the kind of movie that...   

     similarity  
357    0.290119  
316    0.252117  


In [ ]:
import spacy

nlp = spacy.load('en_core_web_sm')

def get_entities(review):
    doc = nlp(review)
    categories = {}
    
    for ent in doc.ents:
        label = ent.label_
        if label not in ["LANGUAGE", "LOC"]: continue
        if label not in categories.keys():
            categories[label] = []
            
        categories[label].append(ent.text)
        
    return categories

In [ ]:
get_entities('English')

{'LANGUAGE': ['English']}

In [ ]:
import pickle

def check_model():
    try:
        with open('model.pkl', 'rb') as f:
            global classifier
            classifier = pickle.load(f)
            print("Model loaded")
    except:
        with open('model.pkl', 'wb') as f:
            print("Training model")
            train_model()
            pickle.dump(classifier, f)

check_model()

Load Model


EOFError: Ran out of input

In [ ]:
def main():
    check_model()
    
    review = ""
    while True:
        print(f"Review: {review}")
        print("1. Write Your Review ")
        print("2. View Movie Recommendation")
        print("3. View Named Entity Recognition")
        print("4. Exit")
        
        choice = input("Input your choice: ")
        
        if choice == '1':
            review = input("Input your review: ")
            print(predict_review(review))
        elif choice == '2':
            if review == '': continue
            get_top_movies(review)
        elif choice == '3':
            if review == "" : continue
            entities = get_entities(review)
            for entity in entities:
                print(f"{entity}: {' '.join(entities[entity])}")
                
        elif choice == '4':
            return        

In [ ]:
main()

Load Model


EOFError: Ran out of input